# Microsoft Fabric: query-aware Warehouse demo

This notebook generates repeatable synthetic data for testing SQL query behavior in a Fabric environment. It does not claim to reproduce production performance.

In [ ]:
%pip install great-generator

In [ ]:
from great_generator import generate_from_schema, validate_query_coverage

schema = """
member_id string,
business_date date,
region string,
product_type string,
member_status string,
interaction_count int,
balance double
"""
required_values = {"region": ["SOUTH"], "product_type": ["CHECKING", "SAVINGS"], "member_status": ["ACTIVE"]}
partition_by = {"column": "business_date", "values": ["2026-01-01", "2026-01-02", "2026-01-03"], "distribution": "balanced"}

pdf = generate_from_schema(schema, rows=100000, required_values=required_values, partition_by=partition_by, seed=42)
report = validate_query_coverage(pdf, required_values=required_values, partition_by=partition_by)
report

sdf = spark.createDataFrame(pdf)
sdf.write.mode("overwrite").format("delta").partitionBy("business_date").saveAsTable("synthetic_query_test")